# 1. Libraries and Data Loading

In [333]:
import numpy as np
import pandas as pd

import sklearn

import plotly.express as px

import os

In [334]:
data = pd.read_csv("../../data/data_raw.csv")
data.head(5)

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories


In [335]:
data.dtypes

week                  str
sku                 int64
weekly_sales      float64
feat_main_page       bool
color                 str
price             float64
vendor              int64
functionality         str
dtype: object

In [336]:
data['week']=pd.to_datetime(data['week'])

In [337]:
data.sort_values(['sku','week'])

data['week'].min(), data['week'].max()

(Timestamp('2016-10-31 00:00:00'), Timestamp('2018-09-24 00:00:00'))

In [338]:
def panel_data_split(data: pd.DataFrame, val_cutoff: int, test_cutoff: int):
    '''
    Splits panel data into train/val/test splits based on cutoff week thresholds.
  2. Sort by [sku, week].
  3. Look at the date range and per-SKU min/max to spot late-starters and gap problems.
  4. Define cutoff_val and cutoff_test (e.g. last 13 weeks test, 13 weeks before that val).
  5. Build train, val, test as filtered DataFrames.
  6. Then do feature engineering — and from this point, any time you write a .mean(), .std(), or any aggregation, ask yourself: "am I computing this from train only?"
    
    return:
    train: training observations, data subset
    val: validation set
    test: test set
    '''
    #sort the data
    data=data.sort_values(['sku','week'])

    #get the number of weeks
    unique_weeks=sorted(data['week'].unique())
    #find index that cutsoff for both test and val
    test_idx = len(unique_weeks) - test_cutoff
    val_idx = test_idx - val_cutoff

    # get date time by index
    test_start_date = unique_weeks[test_idx]
    val_start_date = unique_weeks[val_idx]

    #boolean mask filtering
    train_mask = data['week'] < val_start_date
    val_mask = (data['week'] >= val_start_date) & (data['week'] < test_start_date)
    test_mask = data['week'] >= test_start_date
    #store in train,val,test variables
    train = data[train_mask].copy()
    val = data[val_mask].copy()
    test = data[test_mask].copy()

    return train, val, test


In [339]:
train, val, test = panel_data_split(data, val_cutoff=13, test_cutoff=13)

# 2. EDA & Preprocessing

## 2.1 Summary Stats

In [ ]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4400 entries, 0 to 4399
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   week            4400 non-null   datetime64[us]
 1   sku             4400 non-null   int64         
 2   weekly_sales    4400 non-null   float64       
 3   feat_main_page  4400 non-null   bool          
 4   color           4390 non-null   str           
 5   price           4400 non-null   float64       
 6   vendor          4400 non-null   int64         
 7   functionality   4400 non-null   str           
dtypes: bool(1), datetime64[us](1), float64(2), int64(2), str(2)
memory usage: 245.1 KB


In [ ]:
train.describe()

,week,sku,weekly_sales,price,vendor
count,4400,4400.000000,4400.000000,4400.000000,4400.000000
mean,2017-10-12 12:00:00,22.500000,83.054773,44.432709,6.909091
min,2016-10-31 00:00:00,1.000000,0.000000,2.390000,1.000000
25%,2017-04-22 06:00:00,11.750000,11.000000,15.680000,6.000000
50%,2017-10-12 12:00:00,22.500000,25.000000,27.550000,6.500000
75%,2018-04-03 18:00:00,33.250000,70.000000,54.990000,9.000000
max,2018-09-24 00:00:00,44.000000,7512.000000,227.720000,10.000000
std,NaN,12.699868,288.000205,42.500295,2.503175


### 2.2 Visualisations

In [ ]:
train.isnull().sum()

week               0
sku                0
weekly_sales       0
feat_main_page     0
color             10
price              0
vendor             0
functionality      0
dtype: int64

In [ ]:
train[train.isna().any(axis=1)]

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality
802,2016-11-14,9,54.0,True,NaN,139.44,9,11.Fitness trackers
803,2016-11-21,9,71.0,True,NaN,141.16,9,11.Fitness trackers
4133,2017-06-19,42,4.0,False,NaN,27.33,10,09.Smartphone stands
4196,2018-09-03,42,8.0,False,NaN,42.99,10,09.Smartphone stands
4197,2018-09-10,42,14.0,True,NaN,42.99,10,09.Smartphone stands
4200,2016-10-31,43,5.0,True,NaN,109.99,9,11.Fitness trackers
4314,2017-02-06,44,5.0,False,NaN,53.99,6,09.Smartphone stands
4391,2018-07-30,44,34.0,True,NaN,41.99,6,09.Smartphone stands
4396,2018-09-03,44,14.0,False,NaN,52.99,6,09.Smartphone stands
4398,2018-09-17,44,28.0,True,NaN,42.99,6,09.Smartphone stands


In [ ]:
train['color'].unique()

<StringArray>
[ 'black',   'blue', 'purple',    'red',      nan,  'white',   'none',
  'green',   'grey',   'gold',   'pink']
Length: 11, dtype: str

In [ ]:
train['color'].value_counts()

color
black     1691
blue       700
red        500
green      400
grey       300
white      200
none       200
gold       199
purple     100
pink       100
Name: count, dtype: int64

In [ ]:
train['color'].fillna('none', inplace=True)

/var/folders/8p/3y0vt8nx0j77_stq2p7d95t80000gn/T/ipykernel_34476/1534418620.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['color'].fillna('none', inplace=True)


0       black
1       black
2       black
3       black
4       black
        ...  
4395    black
4396     none
4397    black
4398     none
4399    black
Name: color, Length: 4400, dtype: str

In [ ]:
for index,row in train.iterrows():
    print(row)

week                      2016-10-31 00:00:00
sku                                         1
weekly_sales                            135.0
feat_main_page                           True
color                                   black
price                                   10.16
vendor                                      6
functionality     06.Mobile phone accessories
Name: 0, dtype: object
week                      2016-11-07 00:00:00
sku                                         1
weekly_sales                            102.0
feat_main_page                           True
color                                   black
price                                    9.86
vendor                                      6
functionality     06.Mobile phone accessories
Name: 1, dtype: object
week                      2016-11-14 00:00:00
sku                                         1
weekly_sales                            110.0
feat_main_page                           True
color                             

# 3. Feature Engineering

In [ ]:
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories


## 3.1 Sales Lag

In [ ]:
train['1week_sales_lag']=train.sort_values().groupby('sku')['weekly_sales'].shift(1)
train['2week_sales_lag']=train.sort_values().groupby('sku')['weekly_sales'].shift(2)
train['3week_sales_lag']=train.sort_values().groupby('sku')['weekly_sales'].shift(3)
train['4week_sales_lag']=train.sort_values().groupby('sku')['weekly_sales'].shift(4)

TypeError: DataFrame.sort_values() missing 1 required positional argument: 'by'

## 3.2 Revenue

In [ ]:
train['weekly_revenue']=train['price']*train['weekly_sales']

train['1week_revenue_lag']=train.sort_values().groupby('sku')['weekly_revenue'].shift(1)
train['2week_revenue_lag']=train.sort_values().groupby('sku')['weekly_revenue'].shift(2)
train['3week_revenue_lag']=train.sort_values().groupby('sku')['weekly_revenue'].shift(3)
train['4week_revenue_lag']=train.sort_values().groupby('sku')['weekly_revenue'].shift(4)

In [ ]:
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,4week_sales_lag,weekly_revenue,1week_revenue_lag,4week_revenue_lag
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,1371.60,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,1005.72,1371.60,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,NaN,1126.40,1005.72,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,NaN,1050.29,1126.40,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,135.0,741.72,1050.29,1371.6


## 3.3 % Changes

In [ ]:
train['1week_sales_%_change']=round((train['weekly_sales']-train['1week_sales_lag'])/train['1week_sales_lag'],2)
train['2week_sales_%_change']=round((train['weekly_sales']-train['2week_sales_lag'])/train['2week_sales_lag'],2)
train['3week_sales_%_change']=round((train['weekly_sales']-train['3week_sales_lag'])/train['3week_sales_lag'],2)
train['4week_sales_%_change']=round((train['weekly_sales']-train['4week_sales_lag'])/train['4week_sales_lag'],2)

train['1week_revenue_%_change']=round((train['weekly_revenue']-train['1week_revenue_lag'])/train['1week_revenue_lag'],2)
train['2week_revenue_%_change']=round((train['weekly_revenue']-train['2week_revenue_lag'])/train['2week_revenue_lag'],2)
train['3week_revenue_%_change']=round((train['weekly_revenue']-train['3week_revenue_lag'])/train['3week_revenue_lag'],2)
train['4week_revenue_%_change']=round((train['weekly_revenue']-train['4week_revenue_lag'])/train['4week_revenue_lag'],2)

In [ ]:
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,4week_sales_lag,weekly_revenue,1week_revenue_lag,4week_revenue_lag,1week_sales_%_change,4week_sales_%_change,1week_revenue_%_change,4week_revenue_%_change
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,1371.60,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,1005.72,1371.60,NaN,-0.24,NaN,-0.27,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,NaN,1126.40,1005.72,NaN,0.08,NaN,0.12,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,NaN,1050.29,1126.40,NaN,0.15,NaN,-0.07,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,135.0,741.72,1050.29,1371.6,-0.34,-0.38,-0.29,-0.46


## 3.4 Price

In [ ]:
train['1week_price_lag']=train.groupby(['sku','vendor'])['price'].shift(1)
train['2week_price_lag']=train.groupby(['sku','vendor'])['price'].shift(2)
train['3week_price_lag']=train.groupby(['sku','vendor'])['price'].shift
train['4week_price_lag']=train.groupby(['sku','vendor'])['price'].shift(4)
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,4week_sales_lag,weekly_revenue,1week_revenue_lag,4week_revenue_lag,1week_sales_%_change,4week_sales_%_change,1week_revenue_%_change,4week_revenue_%_change,1week_price_lag,4week_price_lag
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,1371.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,1005.72,1371.60,NaN,-0.24,NaN,-0.27,NaN,10.16,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,NaN,1126.40,1005.72,NaN,0.08,NaN,0.12,NaN,9.86,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,NaN,1050.29,1126.40,NaN,0.15,NaN,-0.07,NaN,10.24,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,135.0,741.72,1050.29,1371.6,-0.34,-0.38,-0.29,-0.46,8.27,10.16


In [ ]:
train['1week_price_%_change']=round((train['price']-train['1week_price_lag'])/train['1week_price_lag'],2)
train['2week_price_%_change']=round((train['price']-train['2week_price_lag'])/train['2week_price_lag'],2)
train['3week_price_%_change']=round((train['price']-train['3week_price_lag'])/train['3week_price_lag'],2)
train['4week_price_%_change']=round((train['price']-train['4week_price_lag'])/train['4week_price_lag'],2)
train.head()

,week,sku,weekly_sales,feat_main_page,color,price,vendor,functionality,1week_sales_lag,4week_sales_lag,...,1week_revenue_lag,4week_revenue_lag,1week_sales_%_change,4week_sales_%_change,1week_revenue_%_change,4week_revenue_%_change,1week_price_lag,4week_price_lag,1week_price_%_change,4week_price_%_change
0,2016-10-31,1,135.0,True,black,10.16,6,06.Mobile phone accessories,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-11-07,1,102.0,True,black,9.86,6,06.Mobile phone accessories,135.0,NaN,...,1371.60,NaN,-0.24,NaN,-0.27,NaN,10.16,NaN,-0.03,NaN
2,2016-11-14,1,110.0,True,black,10.24,6,06.Mobile phone accessories,102.0,NaN,...,1005.72,NaN,0.08,NaN,0.12,NaN,9.86,NaN,0.04,NaN
3,2016-11-21,1,127.0,True,black,8.27,6,06.Mobile phone accessories,110.0,NaN,...,1126.40,NaN,0.15,NaN,-0.07,NaN,10.24,NaN,-0.19,NaN
4,2016-11-28,1,84.0,True,black,8.83,6,06.Mobile phone accessories,127.0,135.0,...,1050.29,1371.6,-0.34,-0.38,-0.29,-0.46,8.27,10.16,0.07,-0.13


## 3.x IsFirst